In [ ]:
%run setup.py

# 04 · Aprendizaje no supervisado — K-Means

**Objetivo:** segmentar viviendas según sus características (sin usar `Price`
como entrada) y describir los perfiles obtenidos.

Para elegir `k` se combinan el **método del codo** y el **silhouette score**,
tal como exige el documento. Luego se analizan tamaños y perfiles de grupo.

In [ ]:
# CELDA 4: Datos para segmentar (solo características)
df = load_processed()
features_cluster = ["Rooms", "BuildingArea", "Distance", "Antiguedad"]
df = df.dropna(subset=features_cluster)

X_cluster = df[features_cluster].copy()
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)
print("Viviendas a segmentar:", X_cluster_scaled.shape)

## 4.0 Fundamento teórico

**K-Means** busca particionar las viviendas en `k` grupos minimizando la
**inercia** (suma de distancias al cuadrado dentro de cada grupo):

`J = Σ(k=1..K) Σ(x ∈ Ck) ||x − μk||²`

El algoritmo alterna dos pasos hasta converger: asignar cada punto al centroide
más cercano y recalcular cada centroide como la media de su grupo.

El **silhouette score** mide qué tan bien separado está cada punto:

`s(i) = (b(i) − a(i)) / max(a(i), b(i))`

donde `a(i)` es la distancia media al propio grupo y `b(i)` al grupo vecino.
Su rango es [-1, 1]; valores más altos indican mejor separación.

## 4.1 Elección del número de grupos: codo y silhouette

In [ ]:
inercias, silhouetas = [], []
rango_k = range(2, 11)

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = km.fit_predict(X_cluster_scaled)
    inercias.append(km.inertia_)
    silhouetas.append(
        silhouette_score(X_cluster_scaled, etiquetas, sample_size=5000, random_state=42)
    )

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(list(rango_k), inercias, marker="o")
axes[0].set_title("Método del codo")
axes[0].set_xlabel("Número de grupos (k)")
axes[0].set_ylabel("Inercia")

axes[1].plot(list(rango_k), silhouetas, marker="o", color="darkorange")
axes[1].set_title("Silhouette score por k")
axes[1].set_xlabel("Número de grupos (k)")
axes[1].set_ylabel("Silhouette")

plt.tight_layout()
plt.show()

tabla_k = pd.DataFrame({"k": list(rango_k), "inercia": inercias, "silhouette": silhouetas})
display(tabla_k.round(3))

In [ ]:
# Se selecciona k=3: el codo se estabiliza y el documento pide tres perfiles
# (económicas, medianas y premium), además de un silhouette aceptable.
K_FINAL = 3
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df["Segmento"] = kmeans.fit_predict(X_cluster_scaled)

print(f"Silhouette con k={K_FINAL}: "
      f"{silhouette_score(X_cluster_scaled, df['Segmento'], sample_size=5000, random_state=42):.4f}")
print("\nTamaño de cada grupo:")
display(df["Segmento"].value_counts().sort_index().to_frame("viviendas"))

## 4.2 Perfilamiento de los segmentos

In [ ]:
perfil = df.groupby("Segmento")[["Price", "Rooms", "BuildingArea", "Distance", "Antiguedad"]].mean()
perfil["viviendas"] = df["Segmento"].value_counts().sort_index()
display(perfil.round(2))

# Etiquetar cada segmento según su precio medio
orden = perfil["Price"].sort_values().index.tolist()
etiquetas = {orden[0]: "Económico", orden[1]: "Mediano", orden[2]: "Premium"}
df["Perfil"] = df["Segmento"].map(etiquetas)
print("Perfiles asignados:", etiquetas)

In [ ]:
plt.figure(figsize=(10, 6))
for segmento in sorted(df["Segmento"].unique()):
    datos = df[df["Segmento"] == segmento]
    plt.scatter(datos["Rooms"], datos["Price"], alpha=0.4, s=15,
                label=f"Segmento {segmento} ({etiquetas[segmento]})")
plt.title("Segmentación de viviendas: habitaciones vs precio")
plt.xlabel("Habitaciones")
plt.ylabel("Precio (AUD)")
plt.legend()
plt.tight_layout()
plt.show()

print("Los grupos se diferencian principalmente por tamaño, superficie y precio.")
print("Limitación: BuildingArea y Antiguedad fueron imputadas, lo que suaviza los perfiles.")